# Milestone 2 — Conditional Generation & Tuning

**Model:** Stable Diffusion v1.5 · **Encoder:** CLIP ViT-L/14 · **Metric:** CLIP score

This notebook is the narrative of milestone 2's four core bullets:

1. **Integrate** the text encoder with the diffusion model — explicitly, through `prompt_embeds`.
2. **Baseline** conditional generation on a fixed prompt set.
3. **Tune** classifier-free guidance, the sampler, and the noise schedule.
4. **Record** the first set of generated images.

Every number below comes from a module under `src/modules/`, each of which is unit-tested on CPU
in `tests/`. The notebook decides *what* to run and shows the result; it does not implement anything.

**No weights are trained here.** "Tune" in bullet 3 means turning knobs, and every core bullet is
satisfied at inference time — which is what makes the whole thing fit in under half an hour on one
consumer GPU.

---
## 0. Kaggle setup

**Skipped entirely off Kaggle**, so this notebook still runs unchanged on a local machine.

On Kaggle it finds whichever attached dataset holds milestone 1's `processed/` output, copies it
somewhere writable, and points `config.py` at it. The copy is not optional: `/kaggle/input` is
read-only and `cfg.ensure_dirs()` creates directories under `DATA_DIR`.

This cell must stay **above the import cell** — `config.py` resolves every path once, when it is
first imported, and setting the variables afterwards has no effect.

In [ ]:
# --- Kaggle paths (no-op anywhere else) ------------------------------------
import os
from pathlib import Path

if Path("/kaggle/input").is_dir():
    import shutil

    attached = sorted(Path("/kaggle/input").iterdir())
    source = next((d for d in attached if (d / "processed" / "pairs.parquet").is_file()), None)
    if source is None:
        raise RuntimeError(
            "No attached dataset contains processed/pairs.parquet -- attach milestone 1's data/ "
            f"directory. Saw: {[d.name for d in attached]}"
        )

    work = Path("/kaggle/working/data")
    if not (work / "processed").is_dir():
        shutil.copytree(source / "processed", work / "processed")

    os.environ["T2I_DATA_DIR"] = str(work)
    os.environ["T2I_OUTPUT_DIR"] = "/kaggle/working/outputs"
    print(f"Kaggle: data from {source.name}, outputs to /kaggle/working/outputs")
else:
    print("Not on Kaggle -- using the repo's own data/ and outputs/ directories.")

In [ ]:
# --- make src/ importable --------------------------------------------------
import sys
from pathlib import Path


def _find_src():
    roots = [Path.cwd(), *Path.cwd().parents]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        roots += sorted(kaggle_input.iterdir())
    for root in roots:
        if (root / "src" / "config.py").is_file():
            return root / "src"
    raise RuntimeError("could not find src/config.py -- is this running inside the repo?")


SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"modules imported from: {SRC}")

---
## 1. Environment

The sweep is memory-bound before it is compute-bound. Stable Diffusion 1.5 in float16 is about
2.5 GB of weights, and each 512×512 image in flight costs roughly another 0.6 GB — so the batch size
is a function of the card, not of taste. `config.GEN_BATCH_SIZE` is set for an 8 GB laptop GPU that is
also driving a display; on a 16 GB datacentre card it can safely double.

In [ ]:
import json, time, warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

import transformers
import diffusers

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()

import config as cfg
import utils
from modules import (clip_embeddings, conditioning, generation, latents, prompts,
                     recording, sanity_checks, sd_pipeline, storage, sweep)

cfg.ensure_dirs()
utils.set_seed()

print("TOOLCHAIN")
print("-" * 52)
for name, version in [("python", sys.version.split()[0]), ("torch", torch.__version__),
                      ("transformers", transformers.__version__), ("diffusers", diffusers.__version__)]:
    print(f"  {name:<16}{version}")

print()
print("HARDWARE")
print("-" * 52)
for key, value in sd_pipeline.vram_report().items():
    print(f"  {key:<16}{value}")
if cfg.DEVICE.type == "cuda":
    major, minor = torch.cuda.get_device_capability(0)
    print(f"  {'compute_cap':<16}{major}.{minor}")
    print(f"  {'kernels_built':<16}{'yes' if f'sm_{major}{minor}' in torch.cuda.get_arch_list() else 'NO -- wrong wheel'}")
print()
print(f"  {'batch size':<16}{cfg.GEN_BATCH_SIZE}")
print(f"  {'SD dtype':<16}{cfg.SD_DTYPE}")

---
## 2. The fixed prompt set

A sweep is only a sweep if the prompts never move. Every configuration below renders **the same ten
captions from the same ten latents**, so a difference in CLIP score is a difference in the knob and
nothing else.

The captions are drawn from milestone 1's curated manifest rather than written by hand — they are real
COCO captions the encoders have already been measured on. The 8–18 word band matters: a three-word
caption underdetermines the image, so every configuration scores well and the sweep separates nothing.

In [ ]:
manifest = storage.read_manifest()
picks = prompts.fixed_prompts(manifest)

PROMPTS = picks["caption"].tolist()
REAL_IMAGES = [cfg.PROJECT_ROOT / p for p in picks["image_512_path"]]

print(f"{len(manifest):,} curated pairs -> {len(PROMPTS)} fixed prompts "
      f"({cfg.PROMPT_MIN_WORDS}-{cfg.PROMPT_MAX_WORDS} words, seed {cfg.SEED})")
print()
for i, (caption, n_words) in enumerate(zip(PROMPTS, picks["n_words"])):
    print(f"  {i:>2}  [{n_words:>2}w]  {caption}")

### The latents

Fixing the seed is **not enough**. Ancestral and SDE samplers draw extra noise inside the denoising
loop, and different samplers consume the RNG stream differently, so two configurations sharing a seed
do not share a starting point. The only way to hold the starting noise still is to generate it once
and pass the tensor in.

It is drawn on a **CPU** generator, because `torch.Generator(device="cuda")` is a different RNG stream
— a CUDA-seeded run does not reproduce on another machine, or on another GPU. Milestone 1 used a CUDA
generator; this corrects it.

In [ ]:
LATENTS = latents.make_latents(len(PROMPTS))

print(f"latents     : {tuple(LATENTS.shape)}  {LATENTS.dtype}  on {LATENTS.device}")
print(f"seed        : {cfg.LATENT_SEED}")
print(f"reproducible: {torch.equal(LATENTS, latents.make_latents(len(PROMPTS)))}")

---
## 3. Integrating the text encoder — task 1

Normally you pass `prompt="a dog"` and the pipeline runs CLIP for you. **Integrating the encoder
explicitly means doing that half ourselves**: load the text encoder, produce the `(N, 77, 768)`
conditioning tensor, and hand it to the UNet. There is no `prompt=` anywhere after this cell.

Per-token states, not the pooled vector — the UNet cross-attends to each token position, which is how
"a *red* bus" colours the bus instead of tinting the whole frame.

The encoder is loaded in float32, used, and **freed before Stable Diffusion loads**. On 8 GB, holding a
1.4 GB encoder you have finished with is 1.4 GB the UNet cannot have. The embeddings themselves are
about a megabyte.

In [ ]:
sd_tokenizer, sd_text_encoder = clip_embeddings.load_sd_text_encoder()

POS, NEG = conditioning.conditioning_pair(PROMPTS, sd_tokenizer, sd_text_encoder)
POS, NEG = POS.cpu(), NEG.cpu()

print(f"encoder      : {cfg.CLIP_SD_MODEL}  ({clip_embeddings.describe(sd_text_encoder)['parameters_millions']:.0f}M params, {cfg.ENCODER_DTYPE})")
print(f"prompt_embeds: {tuple(POS.shape)}  {POS.dtype}")
print(f"negative     : {tuple(NEG.shape)}  (the empty string, broadcast to every prompt)")
print(f"size on disk : {POS.numel() * POS.element_size() / 1024**2:.1f} MB")

sd_pipeline.free(sd_text_encoder)
del sd_text_encoder
print()
print("encoder freed:", sd_pipeline.vram_report())

---
## 4. Loading Stable Diffusion 1.5

The original `runwayml/stable-diffusion-v1-5` repository was withdrawn, so `config.SD_REPO_CANDIDATES`
lists mirrors and they are tried in order.

The scheduler's **original config is captured here**, before anything touches it. Every sampler swap
later rebuilds from this dict rather than from the live scheduler — swapping twice from the live one
compounds the first swap's overrides into the second, and `beta_schedule` silently drifts away from the
`scaled_linear` SD 1.5 was trained on.

In [ ]:
t0 = time.time()
pipe, repo, BASE_SCHEDULER_CONFIG = sd_pipeline.load_sd()
load_seconds = time.time() - t0

print(f"loaded       : {repo}  in {load_seconds:.1f}s")
print(f"scheduler    : {type(pipe.scheduler).__name__}")
print(f"beta_schedule: {BASE_SCHEDULER_CONFIG['beta_schedule']}")
print(f"train steps  : {BASE_SCHEDULER_CONFIG['num_train_timesteps']}")
print()
for key, value in sd_pipeline.vram_report().items():
    print(f"  {key:<16}{value}")

### Verification: does the hand-wired path match the built-in one?

The check that the integration is real. We ask the pipeline to encode the same prompt its own way and
compare, then render the same latent through both paths.

Expect *near*-identical, not bit-identical: our encoder runs in float32 and the pipeline's runs in
float16, so a difference around `1e-3` on the embeddings is the precision gap, not a bug. A difference
of zero would mean we had accidentally called the pipeline's encoder; a large one would mean the
tensors are not what the UNet expects.

In [ ]:
with torch.no_grad():
    pipe_pos, pipe_neg = pipe.encode_prompt(
        prompt=PROMPTS[:1], device=cfg.DEVICE, num_images_per_prompt=1,
        do_classifier_free_guidance=True, negative_prompt=[""])

delta_pos = conditioning.max_abs_difference(POS[:1].to(cfg.DEVICE), pipe_pos)
delta_neg = conditioning.max_abs_difference(NEG[:1].to(cfg.DEVICE), pipe_neg)

print(f"ours   : {tuple(POS[:1].shape)}  {POS.dtype}")
print(f"theirs : {tuple(pipe_pos.shape)}  {pipe_pos.dtype}")
print(f"max |difference|  positive: {delta_pos:.2e}   negative: {delta_neg:.2e}")
print(f"relative to the embedding scale ({POS.abs().max():.2f}): {delta_pos / float(POS.abs().max()):.2e}")

In [ ]:
VERIFY = generation.GenConfig("pndm", cfg.N_STEPS, cfg.GUIDANCE)

ours = generation.generate(pipe, POS[:1], NEG[:1], LATENTS[:1], VERIFY,
                           base_config=BASE_SCHEDULER_CONFIG, batch_size=1)[0]

sd_pipeline.set_scheduler(pipe, VERIFY.sampler, base_config=BASE_SCHEDULER_CONFIG,
                          **dict(VERIFY.scheduler_kwargs))
theirs = pipe(
    prompt=PROMPTS[:1], negative_prompt=[""],
    latents=LATENTS[:1].to(device=pipe.device, dtype=pipe.dtype),
    generator=generation.prompt_generators(1),
    num_inference_steps=VERIFY.steps, guidance_scale=VERIFY.guidance,
).images[0]

pixel_gap = np.abs(np.asarray(ours, float) - np.asarray(theirs, float)).mean()

fig, axes = plt.subplots(1, 2, figsize=(9, 4.8))
for ax, img, title in zip(axes, [ours, theirs], ["prompt_embeds= (ours)", "prompt= (built-in)"]):
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis("off")
fig.suptitle(f"{PROMPTS[0]}\nmean |pixel difference| = {pixel_gap:.2f} / 255", fontsize=9)
fig.tight_layout()
print(utils.save_fig(fig, "m2_integration_check"))
plt.show()

---
## 5. Baseline conditional generation — task 2

The control every tuned configuration is measured against: **PNDM at CFG 7.5**, SD 1.5's own defaults.

It is taken *out of the guidance grid* rather than written separately, so the baseline is literally a
row in the sweep. A baseline that is not a point in any grid cannot be subtracted from the best one —
the difference would mix the sampler, the step count and the schedule together and call the total an
improvement. That also means 50 steps here, PNDM's entry in the sampler table, where milestone 1 used
30; the earlier number is a reference point, not this notebook's control.

The metric is `sanity_checks.clip_scores` from milestone 1, unchanged. It is a cosine in CLIP's joint
space between the caption and the generated image. For reference, the **real COCO photographs** these
captions describe score about 0.318 — that is the number to read the generated scores against, not 1.0.
A text–image cosine above roughly 0.30 means the image matches the caption; 0.1 means it does not.

In [ ]:
clip_tokenizer, clip_joint, clip_processor = clip_embeddings.load_joint()


# The sweep's single metric: CLIP(text, generated), one cosine per prompt.
def scorer(images, prompt_list):
    txt_gen, _, _ = sanity_checks.clip_scores(
        list(prompt_list), images, REAL_IMAGES, clip_tokenizer, clip_joint, clip_processor)
    return txt_gen


BASELINE = next(c for c in sweep.guidance_configs() if c.guidance == cfg.GUIDANCE)

t0 = time.time()
baseline_images = generation.generate(pipe, POS, NEG, LATENTS, BASELINE,
                                      base_config=BASE_SCHEDULER_CONFIG)
baseline_seconds = time.time() - t0

baseline_txt_gen, baseline_txt_real, baseline_gen_real = sanity_checks.clip_scores(
    PROMPTS, baseline_images, REAL_IMAGES, clip_tokenizer, clip_joint, clip_processor)

print(f"config    : {BASELINE.label}")
print(f"time      : {baseline_seconds:.1f}s for {len(PROMPTS)} images "
      f"({baseline_seconds / len(PROMPTS):.2f}s each, {baseline_seconds / len(PROMPTS) / BASELINE.steps:.3f}s per step)")
print()
print(f"CLIP(text, generated) : {baseline_txt_gen.mean():.4f} +/- {baseline_txt_gen.std():.4f}")
print(f"CLIP(text, real photo): {baseline_txt_real.mean():.4f} +/- {baseline_txt_real.std():.4f}   <- the reference")
print(f"CLIP(generated, real) : {baseline_gen_real.mean():.4f} +/- {baseline_gen_real.std():.4f}   <- different scale, compare across rows only")

In [ ]:
print(recording.save_run(baseline_images, PROMPTS, "baseline", rows=2, cols=5)[-1])

fig, axes = plt.subplots(2, 5, figsize=(15, 6.8))
for ax, img, caption, score in zip(axes.ravel(), baseline_images, PROMPTS, baseline_txt_gen):
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{score:.3f}  {caption[:34]}...", fontsize=7)
fig.suptitle(f"Baseline — {BASELINE.label} — CLIP(txt,gen) mean {baseline_txt_gen.mean():.4f}", fontsize=11)
fig.tight_layout()
print(utils.save_fig(fig, "m2_baseline_grid"))
plt.show()

---
## 6. Classifier-free guidance — task 3a

CFG is the dial between "ignore the prompt" and "obey the prompt at any cost". At 1.0 there is no
guidance at all and the image is whatever the UNet felt like; as it rises the image tracks the caption
more closely but saturates and loses detail. The spec's sweet spot is 5–9, and above 12 the
oversaturation needs `guidance_rescale=0.7` to correct — which `sweep.guidance_configs` applies
automatically.

Every configuration here is the same sampler and the same latents. Only the dial moves.

In [ ]:
guidance_grid = sweep.guidance_configs()
print(f"{len(guidance_grid)} configs x {len(PROMPTS)} prompts = {len(guidance_grid) * len(PROMPTS)} images")
for c in guidance_grid:
    print(f"  {c.label}")

guidance_results = sweep.run_sweep(
    pipe, guidance_grid, PROMPTS, POS, NEG, LATENTS, scorer,
    base_config=BASE_SCHEDULER_CONFIG, table_name="guidance_results")

display(guidance_results[["label", "guidance", "guidance_rescale", "clip_score_mean", "clip_score_std", "seconds", "status"]])

In [ ]:
ok = guidance_results[guidance_results["status"] == "ok"]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.errorbar(ok["guidance"], ok["clip_score_mean"], yerr=ok["clip_score_std"],
            marker="o", capsize=4, label="generated")
ax.axhline(baseline_txt_real.mean(), ls="--", c="grey", label=f"real photos ({baseline_txt_real.mean():.3f})")
ax.set_xlabel("guidance_scale"); ax.set_ylabel("CLIP(text, generated)")
ax.set_title("Prompt adherence against classifier-free guidance")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
print(utils.save_fig(fig, "m2_guidance_sweep"))
plt.show()

---
## 7. Samplers — task 3b

Seven samplers, each at **its own step count**. That is deliberate: DPMSolver++ reaches at 20 steps
what PNDM needs 50 for, so running them all at an equal step count would measure the step count rather
than the sampler. The comparison that matters to someone generating images is quality *per unit of
time*, which is what the `seconds` column shows.

`Euler-a` is ancestral — it injects fresh noise at every step and never converges, so its output keeps
changing with the step count where the others settle.

In [ ]:
sampler_grid = sweep.sampler_configs()
sampler_results = sweep.run_sweep(
    pipe, sampler_grid, PROMPTS, POS, NEG, LATENTS, scorer,
    base_config=BASE_SCHEDULER_CONFIG, table_name="sampler_results")

table = sampler_results.copy()
table["s_per_image"] = (table["seconds"] / len(PROMPTS)).round(2)
display(table[["sampler", "steps", "clip_score_mean", "clip_score_std", "seconds", "s_per_image", "status"]]
        .sort_values("clip_score_mean", ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
ok = sampler_results[sampler_results["status"] == "ok"]
ax.scatter(ok["seconds"], ok["clip_score_mean"], s=60)
for _, row in ok.iterrows():
    ax.annotate(f"{row['sampler']} ({row['steps']})", (row["seconds"], row["clip_score_mean"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xlabel(f"seconds for {len(PROMPTS)} images"); ax.set_ylabel("CLIP(text, generated)")
ax.set_title("Quality against cost, one point per sampler")
ax.grid(alpha=0.3)
fig.tight_layout()
print(utils.save_fig(fig, "m2_sampler_tradeoff"))
plt.show()

---
## 8. Noise schedule — task 3c

These are **scheduler config**, not call arguments — they change how the noise ramp is shaped and which
timesteps get visited, so they have to be baked in when the scheduler is constructed.

| Knob | Options | What it does |
|---|---|---|
| `timestep_spacing` | `leading` (SD legacy), `trailing`, `linspace` | which timesteps get visited |
| `rescale_betas_zero_snr` | True / False | fixes "cannot generate a truly dark image" |
| `use_karras_sigmas` | True / False | smoother spacing, helps most at low step counts |

Twelve combinations, one sampler, one guidance scale. Note that CLIP score is a blunt instrument for
the zero-SNR fix in particular: it measures caption adherence, and "can render a genuinely black
image" is a property it barely sees. Read the grids as well as the table.

In [ ]:
schedule_grid = sweep.schedule_configs()
print(f"{len(schedule_grid)} combinations on {cfg.SCHEDULE_SAMPLER} at CFG {cfg.GUIDANCE}")

schedule_results = sweep.run_sweep(
    pipe, schedule_grid, PROMPTS, POS, NEG, LATENTS, scorer,
    base_config=BASE_SCHEDULER_CONFIG, table_name="schedule_results")

display(schedule_results[["scheduler_kwargs", "clip_score_mean", "clip_score_std", "seconds", "status"]]
        .sort_values("clip_score_mean", ascending=False))

---
## 9. DDPM — 1000 steps against 20

DDPM is the original formulation and it is **not a sweep arm**: at ~1000 steps it costs roughly a
hundred times what DPMSolver++ costs for a worse image. Running it once on two prompts is the honest
way to show that — the story is "the samplers above are what made diffusion practical", and it is worth
about four minutes to show rather than assert.

In [ ]:
k = cfg.DDPM_N_PROMPTS
slow = generation.GenConfig("ddpm", cfg.DDPM_STEPS, cfg.GUIDANCE)
fast = generation.GenConfig("dpmpp", cfg.SAMPLERS["dpmpp"], cfg.GUIDANCE)

timings, panels = {}, {}
for config in (slow, fast):
    t0 = time.time()
    panels[config.label] = generation.generate(pipe, POS[:k], NEG[:k], LATENTS[:k], config,
                                               base_config=BASE_SCHEDULER_CONFIG, batch_size=k)
    timings[config.label] = time.time() - t0

for label, seconds in timings.items():
    print(f"{label:<28}{seconds:7.1f}s   ({seconds / k:.1f}s per image)")
print(f"\nspeed-up: {timings[slow.label] / timings[fast.label]:.0f}x for {slow.steps // fast.steps}x fewer steps")

fig, axes = plt.subplots(2, k, figsize=(4.2 * k, 8.8), squeeze=False)
for row, (label, images) in enumerate(panels.items()):
    for col, img in enumerate(images):
        axes[row, col].imshow(img); axes[row, col].axis("off")
    axes[row, 0].set_title(f"{label} — {timings[label]:.0f}s", fontsize=9, loc="left")
fig.tight_layout()
print(utils.save_fig(fig, "m2_ddpm_vs_dpmpp"))
plt.show()

---
## 10. Results — task 4

Everything above, in one table and one picture. The comparison grid puts the baseline and the best
configuration on the same prompts and the same latents, so the difference on the page is the difference
the tuning made.

In [ ]:
all_results = pd.concat([
    guidance_results.assign(axis="guidance"),
    sampler_results.assign(axis="sampler"),
    schedule_results.assign(axis="schedule"),
], ignore_index=True)

# PNDM at CFG 7.5 is a member of both the guidance and the sampler grid -- the same
# config, rendered twice. Keeping both would let one configuration occupy two places in
# one ranking.
duplicates = int(all_results["label"].duplicated().sum())
all_results = all_results.drop_duplicates(subset="label", keep="first").reset_index(drop=True)

ranked = sweep.summarise(all_results)
if ranked.empty:
    raise RuntimeError(
        "every config failed -- nothing to rank. The 'error' column in "
        f"{utils.rel(cfg.SWEEP_DIR / 'all_results.csv')} says why; a CUDA build without "
        "kernels for this GPU fails all of them identically."
    )
best = ranked.iloc[0]

print(f"configs run   : {len(all_results)}  ({(all_results['status'] == 'error').sum()} failed, "
      f"{duplicates} duplicates dropped)")
print(f"total GPU time: {all_results['seconds'].sum() / 60:.1f} min")
print()
print(f"baseline      : {BASELINE.label:<34}{baseline_txt_gen.mean():.4f}")
print(f"best          : {best['label']:<34}{best['clip_score_mean']:.4f}")
print(f"real photos   : {'(reference)':<34}{baseline_txt_real.mean():.4f}")
print(f"improvement   : {best['clip_score_mean'] - baseline_txt_gen.mean():+.4f}")
print()
display(ranked.head(10)[["axis", "label", "clip_score_mean", "clip_score_std", "seconds"]])

In [ ]:
best_config = next(c for c in (*guidance_grid, *sampler_grid, *schedule_grid)
                   if c.label == best["label"])
best_images = generation.generate(pipe, POS, NEG, LATENTS, best_config,
                                  base_config=BASE_SCHEDULER_CONFIG)

grid = recording.comparison_grid({
    f"baseline  {BASELINE.label}  ({baseline_txt_gen.mean():.4f})": baseline_images,
    f"best      {best_config.label}  ({best['clip_score_mean']:.4f})": best_images,
})
grid_path = cfg.SWEEP_DIR / "baseline_vs_best.png"
grid.save(grid_path)
print(utils.rel(grid_path))
grid

In [ ]:
all_results.to_csv(cfg.SWEEP_DIR / "all_results.csv", index=False)

summary = {
    "prompts": PROMPTS,
    "prompt_pair_idx": picks["pair_idx"].tolist(),
    "latent_seed": cfg.LATENT_SEED,
    "repo": repo,
    "integration": {
        "max_abs_embedding_difference": delta_pos,
        "mean_abs_pixel_difference": float(pixel_gap),
        "note": "float32 encoder vs float16 pipeline -- near-identical is the expected result",
    },
    "baseline": {
        "config": BASELINE.label,
        "clip_text_generated": float(baseline_txt_gen.mean()),
        "clip_text_real": float(baseline_txt_real.mean()),
        "seconds": round(baseline_seconds, 1),
    },
    "best": {"config": best["label"], "clip_text_generated": float(best["clip_score_mean"])},
    "ddpm_illustration": {k_: round(v, 1) for k_, v in timings.items()},
    "configs_run": int(len(all_results)),
    "configs_failed": int((all_results["status"] == "error").sum()),
    "total_gpu_minutes": round(all_results["seconds"].sum() / 60, 1),
    "hardware": sd_pipeline.vram_report(),
}

print(utils.save_table("milestone_2_sd15", summary))
print(utils.rel(cfg.SWEEP_DIR / "all_results.csv"))
print()
print(json.dumps(summary["baseline"] | summary["best"], indent=2))

### Deliverables from this notebook

| Milestone 2 deliverable | Where |
|---|---|
| Explicit `prompt_embeds` integration, verified against `prompt=` | §3, §4 |
| Baseline conditional generation on the fixed 10-prompt set | §5 |
| CFG sweep (1–15) with CLIP score per setting | §6 |
| Sampler comparison across seven samplers, fixed latents | §7 |
| Noise-schedule comparison | §8 |
| DDPM illustration — 1000 steps vs 20 | §9 |
| First set of generated images recorded, with the results table | §5, §10, `outputs/sweep/` |

The GPT-2 encoder row is in `milestone_2_encoders.ipynb`; aMUSEd and LDM-BERT are in
`milestone_2_comparison.ipynb`.

In [ ]:
sd_pipeline.free(pipe, clip_joint)
del pipe, clip_joint
print("released:", sd_pipeline.vram_report())